<a href="https://colab.research.google.com/github/Navya40869/edge-idps-colab/blob/main/models/02_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("Is GPU available?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

Is GPU available?: True
GPU Device Name: Tesla T4


In [ ]:
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# Load preprocessed arrays directly from Drive
save_path = "/content/drive/MyDrive/Edge-IDPS-Data/processed_data/"
X_train = np.load(save_path + "X_train.npy")
y_train = np.load(save_path + "y_train.npy")
X_test = np.load(save_path + "X_test.npy")
y_test = np.load(save_path + "y_test.npy")

print(f"Loaded X_train shape: {X_train.shape}")

Mounted at /content/drive
Loaded X_train shape: (71538, 39)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Check GPU connectivity
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Convert NumPy arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create PyTorch DataLoaders for mini-batch processing
batch_size = 128
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")

Using device: cuda
Number of training batches: 559


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode labels to be contiguous integers starting from 0 (e.g., 0, 1, 2, ..., N-1)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

# Verify updated class parameters
num_classes = len(label_encoder.classes_)
print(f"Number of target classes: {num_classes}")
print(f"Unique encoded labels in y_train: {np.unique(y_train)}")

Number of target classes: 29
Unique encoded labels in y_train: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28]


In [2]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 1. MOUNT DRIVE & LOAD PREPROCESSED DATA
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/Edge-IDPS-Data/processed_data/"

print("Loading dataset arrays...")
X_train = np.load(save_path + "X_train.npy")
X_test = np.load(save_path + "X_test.npy")
y_train = np.load(save_path + "y_train.npy")
y_test = np.load(save_path + "y_test.npy")

# Encode string/non-contiguous labels to contiguous integers [0, num_classes - 1]
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Save updated label encoder for Member 3 & 5
joblib.dump(label_encoder, save_path + "label_encoder.pkl")
print(f"Dataset loaded! Features: {X_train.shape[1]} | Classes: {len(label_encoder.classes_)}")

# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# ==========================================
# 2. DEFINE 1D-CNN ARCHITECTURE
# ==========================================
class EdgeIDPS1DCNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(EdgeIDPS1DCNN, self).__init__()

        # 1D Convolutional Feature Extractor
        self.conv_block = nn.Sequential(
            # Layer 1: Input shape (batch, 1, features) -> (batch, 32, features)
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            # Layer 2: Output shape (batch, 64, features / 2)
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) # Aggregates features down to 1D vector of length 64
        )

        # Fully Connected Classification Head
        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # Reshape tabular input from (batch, features) to 1D channel input (batch, 1, features)
        x = x.unsqueeze(1)
        x = self.conv_block(x)
        x = self.fc_block(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train.shape[1]
num_classes = len(label_encoder.classes_)

model = EdgeIDPS1DCNN(input_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Training 1D-CNN Model on: {device}")

# ==========================================
# 3. TRAIN THE MODEL
# ==========================================
epochs = 10
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    print(f"Epoch [{epoch}/{epochs}] - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.2f}%")

# ==========================================
# 4. EVALUATE & SAVE MODEL WEIGHTS
# ==========================================
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(labels.numpy())

overall_accuracy = accuracy_score(all_targets, all_preds)
print(f"\n✅ 1D-CNN Final Test Accuracy: {overall_accuracy * 100:.2f}%\n")

# Convert class labels to string list to prevent TypeError
target_names_str = [str(c) for c in label_encoder.classes_]

print("Classification Report:")
print(classification_report(all_targets, all_preds, target_names=target_names_str, zero_division=0))

# Save PyTorch Model Weights
torch.save(model.state_dict(), save_path + "edge_idps_model.pth")
print(f"Saved trained 1D-CNN weights to: {save_path}edge_idps_model.pth")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading dataset arrays...
Dataset loaded! Features: 39 | Classes: 29
Training 1D-CNN Model on: cuda
Epoch [1/10] - Loss: 0.9644 - Accuracy: 72.18%
Epoch [2/10] - Loss: 0.2905 - Accuracy: 87.06%
Epoch [3/10] - Loss: 0.2560 - Accuracy: 87.74%
Epoch [4/10] - Loss: 0.2408 - Accuracy: 88.05%
Epoch [5/10] - Loss: 0.2370 - Accuracy: 88.13%
Epoch [6/10] - Loss: 0.2303 - Accuracy: 88.20%
Epoch [7/10] - Loss: 0.2254 - Accuracy: 88.45%
Epoch [8/10] - Loss: 0.2221 - Accuracy: 88.42%
Epoch [9/10] - Loss: 0.2196 - Accuracy: 88.47%
Epoch [10/10] - Loss: 0.2187 - Accuracy: 88.46%

✅ 1D-CNN Final Test Accuracy: 88.73%

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.95      1.00      0.98       664
           2       0.00      0.00      0.00         4
           3       0.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# 1. Calculate class weights based on y_train distribution
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert weights to tensor and pass to GPU device
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

# 2. Update Loss Function with weights
criterion = nn.CrossEntropyLoss(weight=weights_tensor)